# 0.0 Imports

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import ElasticNet
from sklearn import metrics as mt

# 0.1 - Load Datasets

In [8]:
actual_folder = Path.cwd()
print(actual_folder)

main_folder = actual_folder.parent.parent
print(main_folder)

path_folder_dataset = main_folder / 'dataset' / 'regression_datasets'

if path_folder_dataset.exists():
    print(f'Caminho existente: {path_folder_dataset}')
else:
    print(f'Erro: O arquivo não foi encontrado {path_folder_dataset}')

d:\repos\projetos\ml_trials_algorithm\notebooks\regressao
d:\repos\projetos\ml_trials_algorithm
Caminho existente: d:\repos\projetos\ml_trials_algorithm\dataset\regression_datasets


In [9]:
#Carrega os datasets da pasta - Treinamento / Validação e Teste

dataset_traning_X = path_folder_dataset / 'a_traninig' / 'X_training.csv'
dataset_traning_y = path_folder_dataset / 'a_traninig' / 'y_training.csv'

dataset_validation_X = path_folder_dataset / 'b_validation' / 'X_validation.csv'
dataset_validation_y = path_folder_dataset / 'b_validation' / 'y_validation.csv'

dataset_test_X = path_folder_dataset / 'c_test' / 'X_test.csv'
dataset_test_y = path_folder_dataset / 'c_test' / 'y_test.csv'

X_train = pd.read_csv(dataset_traning_X)
y_train = pd.read_csv(dataset_traning_y)

X_val = pd.read_csv(dataset_validation_X)
y_val = pd.read_csv(dataset_validation_y)

X_test = pd.read_csv(dataset_test_X)
y_test = pd.read_csv(dataset_test_y)

# 1.0 - Training Algorithm

## Passo 1 — Treino com parâmetros default

In [10]:
# Instanciar o modelo com parâmetros default
model_def = ElasticNet()

# Treinar com dados de treino
model_def.fit(X_train, y_train.values.ravel())

# Predições no treino e na validação
yhat_train_def = model_def.predict(X_train)
yhat_val_def   = model_def.predict(X_val)

## Passo 2 — Performance no treino (default)

In [11]:
# Métricas no conjunto de TREINO com parâmetros default
r2_train_def   = mt.r2_score(y_train, yhat_train_def)
mse_train_def  = mt.mean_squared_error(y_train, yhat_train_def)
rmse_train_def = np.sqrt(mse_train_def)
mae_train_def  = mt.mean_absolute_error(y_train, yhat_train_def)
mape_train_def = mt.mean_absolute_percentage_error(y_train, yhat_train_def)

print("--- Performance no Treino (Default) ---")
print(f"R²:   {r2_train_def:.4f}")
print(f"MSE:  {mse_train_def:.2f}")
print(f"RMSE: {rmse_train_def:.2f}")
print(f"MAE:  {mae_train_def:.2f}")
print(f"MAPE: {mape_train_def * 100:.2f}%")

--- Performance no Treino (Default) ---
R²:   0.0078
MSE:  474.27
RMSE: 21.78
MAE:  17.30
MAPE: 873.23%


## Passo 3 — Performance na validação (default)

In [12]:
# Métricas no conjunto de VALIDAÇÃO com parâmetros default
r2_val_def   = mt.r2_score(y_val, yhat_val_def)
mse_val_def  = mt.mean_squared_error(y_val, yhat_val_def)
rmse_val_def = np.sqrt(mse_val_def)
mae_val_def  = mt.mean_absolute_error(y_val, yhat_val_def)
mape_val_def = mt.mean_absolute_percentage_error(y_val, yhat_val_def)

print("--- Performance na Validação (Default) ---")
print(f"R²:   {r2_val_def:.4f}")
print(f"MSE:  {mse_val_def:.2f}")
print(f"RMSE: {rmse_val_def:.2f}")
print(f"MAE:  {mae_val_def:.2f}")
print(f"MAPE: {mape_val_def * 100:.2f}%")

--- Performance na Validação (Default) ---
R²:   0.0081
MSE:  473.64
RMSE: 21.76
MAE:  17.26
MAPE: 869.40%


## Passo 4 — Ajuste de hiperparâmetros

In [13]:
print("--- Testando Múltiplos Hiperparâmetros na Elastic Net ---")
melhor_alpha = 1.0
melhor_l1_ratio = 0.5
melhor_max_iter = 1000
melhor_r2_val = -999

list_alpha = [0.0001, 0.001, 0.01, 0.1, 1.0]
list_l1_ratio = [0.1, 0.3, 0.5, 0.7, 0.9]
list_max_iter = [1000, 2000, 3000]

for alpha in list_alpha:
    for l1_rat in list_l1_ratio:
        for mi in list_max_iter:
        
            model = ElasticNet(alpha=alpha, l1_ratio=l1_rat, max_iter=mi, random_state=42)
        
            try:
                model.fit(X_train, y_train.values.ravel())
            
                yhat_train = model.predict(X_train)
                yhat_val = model.predict(X_val)
            
                r2_train = mt.r2_score(y_train, yhat_train)
                r2_val = mt.r2_score(y_val, yhat_val)
                rmse_val = np.sqrt(mt.mean_squared_error(y_val, yhat_val))
                mae_val = mt.mean_absolute_error(y_val, yhat_val)
            
                mape_raw = mt.mean_absolute_percentage_error(y_val, yhat_val)
                mape_print = f"{mape_raw * 100:.2f}%" if np.isfinite(mape_raw) and mape_raw < 1000 else "Distort (High)"
            
                print(f"Alpha: {alpha:<6} | L1_Ratio: {l1_rat} | Max_Iter: {mi:4d} | R² Treino: {r2_train:.4f} | R² Val: {r2_val:.4f} | RMSE Val: {rmse_val:.2f} | MAPE Val: {mape_print}")
            
                if r2_val > melhor_r2_val:
                    melhor_r2_val = r2_val
                    melhor_alpha = alpha
                    melhor_l1_ratio = l1_rat
                    melhor_max_iter = mi
                
            except Exception as e:
                print(f"Erro real na combinação Alpha {alpha} | L1_Ratio {l1_rat} | Max_Iter {mi}: {e}")

print("=" * 80)
print(f"-> VENCEDOR DO ENSAIO ELASTIC NET:")
print(f"Melhor Alpha: {melhor_alpha}")
print(f"Melhor L1_Ratio: {melhor_l1_ratio}")
print(f"Melhor Max_Iter: {melhor_max_iter}")
print(f"Maior R² obtido na Validação: {melhor_r2_val:.4f}\n")

--- Testando Múltiplos Hiperparâmetros na Elastic Net ---
Alpha: 0.0001 | L1_Ratio: 0.1 | Max_Iter: 1000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.24%
Alpha: 0.0001 | L1_Ratio: 0.1 | Max_Iter: 2000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.24%
Alpha: 0.0001 | L1_Ratio: 0.1 | Max_Iter: 3000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.24%
Alpha: 0.0001 | L1_Ratio: 0.3 | Max_Iter: 1000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.25%
Alpha: 0.0001 | L1_Ratio: 0.3 | Max_Iter: 2000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.25%
Alpha: 0.0001 | L1_Ratio: 0.3 | Max_Iter: 3000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.25%
Alpha: 0.0001 | L1_Ratio: 0.5 | Max_Iter: 1000 | R² Treino: 0.0461 | R² Val: 0.0399 | RMSE Val: 21.41 | MAPE Val: 868.25%
Alpha: 0.0001 | L1_Ratio: 0.5 | Max_Iter: 2000 | R² Treino: 0.0461 | R² Val: 0.0399 | RM

## Passo 5 — Performance com modelo tunado (treino e validação)

In [14]:
# Instanciar o modelo tunado com os melhores hiperparâmetros encontrados no Passo 4
model_tunado = ElasticNet(alpha=melhor_alpha, l1_ratio=melhor_l1_ratio, max_iter=melhor_max_iter, random_state=42)

# Treinar apenas em X_train (sem X_val — avaliação justa antes da união)
model_tunado.fit(X_train, y_train.values.ravel())

yhat_train_tunado = model_tunado.predict(X_train)
yhat_val_tunado   = model_tunado.predict(X_val)

r2_train_tunado   = mt.r2_score(y_train, yhat_train_tunado)
mse_train_tunado  = mt.mean_squared_error(y_train, yhat_train_tunado)
rmse_train_tunado = np.sqrt(mse_train_tunado)
mae_train_tunado  = mt.mean_absolute_error(y_train, yhat_train_tunado)
mape_train_tunado = mt.mean_absolute_percentage_error(y_train, yhat_train_tunado)

r2_val_tunado   = mt.r2_score(y_val, yhat_val_tunado)
mse_val_tunado  = mt.mean_squared_error(y_val, yhat_val_tunado)
rmse_val_tunado = np.sqrt(mse_val_tunado)
mae_val_tunado  = mt.mean_absolute_error(y_val, yhat_val_tunado)
mape_val_tunado = mt.mean_absolute_percentage_error(y_val, yhat_val_tunado)

print("--- Performance com Modelo Tunado ---")
print(f"MÉTRICAS TREINAMENTO (Tunado):")
print(f"R²:   {r2_train_tunado:.4f}")
print(f"RMSE: {rmse_train_tunado:.2f}")
print(f"MAE:  {mae_train_tunado:.2f}")
print(f"MAPE: {mape_train_tunado * 100:.2f}%")
print("=" * 65)
print(f"MÉTRICAS VALIDAÇÃO (Tunado):")
print(f"R²:   {r2_val_tunado:.4f}")
print(f"RMSE: {rmse_val_tunado:.2f}")
print(f"MAE:  {mae_val_tunado:.2f}")
print(f"MAPE: {mape_val_tunado * 100:.2f}%")

--- Performance com Modelo Tunado ---
MÉTRICAS TREINAMENTO (Tunado):
R²:   0.0460
RMSE: 21.35
MAE:  17.00
MAPE: 865.47%
MÉTRICAS VALIDAÇÃO (Tunado):
R²:   0.0399
RMSE: 21.41
MAE:  17.04
MAPE: 868.21%


## Passo 6 — União treino + validação

In [15]:
# Unir treino + validação para formar o conjunto final de treinamento
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"y_train_final shape: {y_train_final.shape}")

X_train_final shape: (15068, 13)
y_train_final shape: (15068, 1)


## Passo 7 — Retreino com melhores parâmetros

In [16]:
# Retreinar com os melhores hiperparâmetros usando o conjunto completo (treino + validação)
model = ElasticNet(alpha=melhor_alpha, l1_ratio=melhor_l1_ratio, max_iter=melhor_max_iter, random_state=42)
model.fit(X_train_final, y_train_final.values.ravel())
print("Modelo final retreinado com sucesso em X_train_final.")

Modelo final retreinado com sucesso em X_train_final.


## Passo 8 — Performance no teste

In [17]:
# Previsão final usando o dataset de teste (que ficou isolado o tempo todo)
yhat_test = model.predict(X_test)

r2_test   = mt.r2_score(y_test, yhat_test)
mse_test  = mt.mean_squared_error(y_test, yhat_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mt.mean_absolute_error(y_test, yhat_test)
mape_test = mt.mean_absolute_percentage_error(y_test, yhat_test)

print(f"MÉTRICAS NO DATASET DE TESTE:")
print(f"R² (Coef. Determinação): {r2_test:.4f}")
print(f"MSE (Erro Quadrático Médio): {mse_test:.2f}")
print(f"RMSE (Raiz do Erro Quadrático): {rmse_test:.2f}")
print(f"MAE (Erro Absoluto Médio): {mae_test:.2f}")
print(f"MAPE (Erro Percentual Médio): {mape_test * 100:.2f}%")
print("=" * 65)

MÉTRICAS NO DATASET DE TESTE:
R² (Coef. Determinação): 0.0511
MSE (Erro Quadrático Médio): 462.01
RMSE (Raiz do Erro Quadrático): 21.49
MAE (Erro Absoluto Médio): 17.14
MAPE (Erro Percentual Médio): 853.71%


## Passo 9 — Quadro Comparativo — Diagnóstico Completo

In [18]:
data_comparativo = {
    'Conjunto': ['Treino (Default)', 'Validação (Default)', 'Treino (Tunado)', 'Validação (Tunado)', 'Teste (Final)'],
    'R²':    [r2_train_def,   r2_val_def,   r2_train_tunado,   r2_val_tunado,   r2_test],
    'RMSE':  [rmse_train_def, rmse_val_def, rmse_train_tunado, rmse_val_tunado, rmse_test],
    'MAE':   [mae_train_def,  mae_val_def,  mae_train_tunado,  mae_val_tunado,  mae_test],
    'MAPE':  [f'{mape_train_def*100:.2f}%', f'{mape_val_def*100:.2f}%',
              f'{mape_train_tunado*100:.2f}%', f'{mape_val_tunado*100:.2f}%',
              f'{mape_test*100:.2f}%'],
}
df_comparativo = pd.DataFrame(data_comparativo)
print("\n--- Quadro Comparativo — Diagnóstico Completo ---")
print(df_comparativo.to_string(index=False))


--- Quadro Comparativo — Diagnóstico Completo ---
           Conjunto       R²      RMSE       MAE    MAPE
   Treino (Default) 0.007832 21.777715 17.299507 873.23%
Validação (Default) 0.008117 21.763171 17.262903 869.40%
    Treino (Tunado) 0.046039 21.354279 16.998787 865.47%
 Validação (Tunado) 0.039940 21.411204 17.038384 868.21%
      Teste (Final) 0.051129 21.494329 17.142742 853.71%
